# Sesión 1: Fundamentos de AIDP
### Preparación de datos de jugadores de fútbol en capas Bronze, Silver y Gold

En esta sesión vamos a construir un flujo sencillo de preparación de datos dentro de AIDP. Usaremos un archivo CSV con información de jugadores y lo transformaremos progresivamente hasta obtener tablas listas para análisis.

Durante el notebook aprenderás a:
- Descargar un archivo CSV desde Object Storage.
- Cargar los datos en pandas y convertirlos a Spark.
- Crear una tabla en la capa **Bronze** con datos crudos.
- Revisar calidad básica de datos: nulos, valores inválidos y primeras filas.
- Refinar los datos en la capa **Silver**.
- Crear variables derivadas, como edad calculada.
- Preparar tablas de la capa **Gold** para consumo analítico.

> Recomendación para la sesión: ejecuta las celdas en orden y revisa la salida antes de continuar. El objetivo es entender cómo fluye la información desde datos crudos hasta datos listos para generar valor.

Para ejecutar cada celda del notebook, haz clic en el botón **Play** o usa el atajo **Ctrl + Enter**.

---

## Capa Bronze

La capa Bronze conserva los datos en una forma muy cercana al origen. Es útil para trazabilidad, auditoría y primeras revisiones de calidad.

En esta primera parte vamos a:
- Descargar el CSV fuente.
- Estandarizar nombres de columnas.
- Crear una tabla Spark en el catálogo de AIDP.
- Guardar los datos crudos para usarlos en las siguientes capas.


### Lectura del CSV desde Object Storage

Comenzamos descargando el archivo CSV desde un bucket público de Object Storage. Luego lo guardamos temporalmente dentro del entorno del notebook y lo cargamos con pandas.

Este paso simula una ingesta simple desde una fuente externa hacia el ambiente de trabajo.


In [1]:
import requests
import pandas as pd

url = "https://objectstorage.us-chicago-1.oraclecloud.com/n/idajmumkp9ca/b/DeepDiveWorkshopData/o/fifa_players.csv"

response = requests.get(url)

with open("/tmp/fifa_players.csv", "wb") as f:
    f.write(response.content)

csv = pd.read_csv("/tmp/fifa_players.csv")

### Conversión de tipos Spark a tipos SQL

Para crear una tabla en el catálogo necesitamos traducir los tipos de datos de Spark a tipos SQL. La siguiente función hace ese mapeo de forma simple.

Esto nos permite generar automáticamente una instrucción `CREATE TABLE` con la estructura del archivo cargado.


In [2]:
def spark_to_spark_sql_type(spark_type):
    t = str(spark_type)
    
    if "StringType" in t:
        return "STRING"
    elif "IntegerType" in t:
        return "INT"
    elif "LongType" in t:
        return "BIGINT"
    elif "DoubleType" in t:
        return "DOUBLE"
    elif "FloatType" in t:
        return "FLOAT"
    elif "BooleanType" in t:
        return "BOOLEAN"
    elif "DateType" in t:
        return "DATE"
    elif "TimestampType" in t:
        return "TIMESTAMP"
    else:
        return "STRING"

### Generación automática del DDL

El DDL es la instrucción SQL que define la estructura de una tabla. En esta celda creamos una función que recorre el esquema del DataFrame de Spark y genera un `CREATE TABLE` con las mismas columnas y tipos.

Así evitamos escribir manualmente la definición de cada columna.


In [4]:
def generate_create_table(df, table_name):
    columns = []
    
    for field in df.schema.fields:
        col_name = field.name
        col_type = spark_to_spark_sql_type(field.dataType)
        columns.append(f"{col_name} {col_type}")
    
    cols_sql = ",\n  ".join(columns)
    
    ddl = f"""
    CREATE TABLE {table_name} (
      {cols_sql}
    )
    """
    
    return ddl

### Ingesta en la capa Bronze

Ahora vamos a preparar los datos para guardarlos como tabla Bronze. Antes de escribirlos, haremos una limpieza mínima de nombres de columnas para evitar problemas con espacios, símbolos o caracteres especiales.

En esta etapa realizaremos cuatro acciones:
- Convertir el CSV de pandas a un DataFrame de Spark.
- Limpiar nombres de columnas.
- Crear la tabla en el esquema disponible de AIDP.
- Escribir los datos en la tabla creada.

> Nota: la capa Bronze no busca corregir todo el dataset. Su objetivo principal es conservar el dato original de manera consultable y organizada.


### Limpieza básica de nombres de columnas

La siguiente celda estandariza los nombres de columnas para que sean compatibles con Spark SQL. Por ejemplo, convierte texto a minúsculas y elimina caracteres que podrían causar errores al crear la tabla.


In [5]:
import re

def clean_col(c):
    return re.sub(r'[^a-zA-Z0-9_]', '', c.lower())

# Estandarizar nombres de columnas para que sean compatibles con Spark SQL.
csv.columns = [clean_col(c) for c in csv.columns]

# Convertir el DataFrame de pandas a un DataFrame de Spark.
spark_df = spark.createDataFrame(csv)

table_name = "bronze_fifa_players"
ddl = generate_create_table(spark_df, table_name)

schema_ok = None
try:
    spark.sql("USE deepdivecatalog_bronze.admin")
    schema_ok = "admin"
    print("Usando esquema 'admin'")
except Exception as e:
    print(f"Esquema 'admin' no encontrado: {e}")
    try:
        spark.sql("USE deepdivecatalog_bronze.oraclelabs")
        schema_ok = "oraclelabs"
        print("Usando esquema 'oraclelabs'")
    except Exception as e2:
        print(f"Tampoco se pudo usar 'oraclelabs': {e2}")
        raise

spark.sql(f"DROP TABLE IF EXISTS {table_name}")
spark.sql(ddl)

spark_df.write     .mode("append")     .saveAsTable(table_name)


Usando esquema 'admin'


### Visualización inicial de la tabla Bronze

Después de guardar la tabla, mostramos algunas filas para confirmar que la carga fue exitosa y que los datos se pueden consultar desde Spark.


In [6]:
df = spark.table("bronze_fifa_players")
display(df.limit(5))

### Análisis inicial de calidad de datos

Antes de transformar los datos, revisaremos si existen valores nulos en las columnas. Esta verificación ayuda a identificar campos incompletos que podrían afectar análisis posteriores.


In [7]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

### Revisión de valores nulos e inválidos

Aunque una columna no tenga valores `Null`, también puede contener valores inválidos como `NaN`. Por eso hacemos una segunda revisión combinando ambos casos.

El resultado ayuda a decidir qué columnas requieren más atención. Por ejemplo, si la información de valor o salario de algunos jugadores está incompleta, los análisis financieros pueden ser menos concluyentes. En cambio, columnas menos críticas para el objetivo del ejercicio pueden revisarse con menor prioridad.


In [8]:
from pyspark.sql.functions import isnan, col, sum

display(df.select([
    sum((col(c).isNull() | isnan(col(c))).cast("int")).alias(c)
    for c in df.columns
]))

---

## Capa Silver

La capa Silver contiene datos más limpios, estandarizados y confiables que la capa Bronze. Aquí empezamos a corregir problemas básicos y a preparar el dataset para análisis.

En esta sección vamos a:
- Leer la tabla Bronze desde el catálogo.
- Convertir la fecha de nacimiento a formato fecha.
- Eliminar duplicados simples por jugador.
- Reemplazar valores inválidos.
- Filtrar registros sin nombre.

Estas transformaciones no cambian el significado del dataset; solo lo dejan en una forma más consistente para trabajar.


In [9]:
from pyspark.sql.functions import col, to_date

bronze_df = spark.table(f"deepdivecatalog_bronze.{schema_ok}.bronze_fifa_players")

silver_df = (bronze_df
    .withColumn("birth_date", to_date("birth_date", "M/d/yyyy"))
    .dropDuplicates(["full_name"])  # Eliminar registros duplicados del mismo jugador.
    .na.replace(float("nan"), None)  # Reemplazar valores NaN por nulos.
    .filter(col("name").isNotNull()) # Conservar registros con nombre de jugador.
)


### Desarrollo de columnas derivadas
#### Edad calculada

El dataset puede contener una columna de edad que ya no está actualizada. Como también tenemos la fecha de nacimiento, podemos calcular una edad aproximada usando la fecha actual del entorno.

Este es un ejemplo de columna derivada: una variable nueva creada a partir de datos existentes.


In [10]:
from pyspark.sql.functions import current_date, year

silver_df = silver_df.withColumn(
    "calculated_age",
    year(current_date()) - year(col("birth_date"))
)

#### Comparación entre edad original y edad calculada

Ahora compararemos la edad incluida en el dataset con la edad calculada. Esta diferencia ayuda a detectar desactualización o inconsistencias en los datos originales.


In [11]:
from pyspark.sql.functions import col, current_date, months_between, abs

silver_df.select(
    "name",
    "age",
    "calculated_age",
    abs(col("age") - col("calculated_age")).alias("age_diff")
).show(5, truncate=False)

+-------------------+---+--------------+--------+
|name               |age|calculated_age|age_diff|
+-------------------+---+--------------+--------+
|Antônio Chiamuloira|31 |38            |7       |
|A. Halme           |20 |28            |8       |
|A. Hughes          |39 |47            |8       |
|A. Lennon          |31 |39            |8       |
|A. Amadi-Holloway  |26 |33            |7       |
+-------------------+---+--------------+--------+
only showing top 5 rows



### Guardado de la tabla Silver

Una vez aplicadas las transformaciones básicas, guardamos el DataFrame procesado como una tabla de la capa Silver.

Esta tabla será la base para construir resultados más orientados al negocio en la siguiente sección.


In [12]:
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("deepdivecatalog_prata.prata_fifa_players")

---

## Capa Gold

La capa Gold contiene datos listos para consumo analítico: reportes, dashboards, métricas de negocio o modelos de machine learning.

En esta sección crearemos una versión más enfocada del dataset, con columnas relevantes y una categoría de calificación para los jugadores.

#### Creación de categorías de calificación

Clasificaremos a los jugadores según su `overall_rating` para facilitar análisis por segmento.


In [13]:
from pyspark.sql.functions import when
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "overall_rating",
    col("overall_rating").cast("int")
)

gold_df = silver_df.withColumn(
    "rating_category",
    when(col("overall_rating") >= 85, "elite")
    .when(col("overall_rating") >= 75, "bueno")
    .otherwise("comun")
)


#### Selección de columnas relevantes

Para la tabla Gold no necesitamos conservar todas las columnas del origen. Seleccionamos solo las variables más útiles para análisis de jugadores, valor, salario y potencial.


In [14]:
gold_df = gold_df.select(
    "name",
    "full_name",
    "birth_date",
    "calculated_age",
    "nationality",
    "overall_rating",
    "potential",
    "value_euro",
    "wage_euro",
    "preferred_foot",
    "rating_category"
)

#### Guardado de la tabla consolidada Gold

Guardamos la tabla principal de la capa Gold. Esta tabla resume la información más importante de cada jugador y puede ser usada por herramientas analíticas.


In [15]:
gold_df.write.mode("overwrite").saveAsTable(
    "deepdivecatalog_ouro.ouro_fifa_players"
)

### Creación de vistas analíticas en la capa Gold
#### Mejores jugadores por calificación y valor

A partir de la tabla Gold consolidada, construiremos consultas derivadas que responden preguntas de negocio simples. La primera muestra los jugadores con mayor calificación general.


In [16]:
gold_top_players = gold_df     .select("name", "nationality", "overall_rating", "value_euro")     .orderBy(col("overall_rating").desc())

display(gold_top_players.limit(10))


### Valor promedio por país

Esta vista calcula el valor promedio de jugadores por nacionalidad. Puede servir como punto de partida para comparar mercados o perfiles de jugadores por país.


In [17]:
from pyspark.sql.functions import avg

gold_value_by_country = gold_df \
    .groupBy("nationality") \
    .agg(avg("value_euro").alias("avg_value_euro"))

display(gold_value_by_country)

### Jugadores con mayor potencial

Esta vista filtra jugadores con potencial alto. Es útil para identificar perfiles prometedores que podrían tener buen desempeño futuro.


In [18]:
gold_high_potential = gold_df \
    .filter(col("potential") > 85) \
    .select("name", "potential", "value_euro")  

gold_high_potential.head()

Row(name='A. Diallo', potential=86, value_euro=15000000.0)

### Eficiencia: calificación frente a salario

Esta vista calcula una relación simple entre valor y salario. El objetivo es explorar jugadores que podrían ofrecer buen valor relativo frente a su costo salarial.


In [19]:
gold_efficiency = gold_df     .withColumn("value_per_wage", col("value_euro") / col("wage_euro"))     .select("name", "value_per_wage", "overall_rating")

display(gold_efficiency.limit(10))


### Guardado de tablas analíticas Gold

Finalmente guardamos las vistas analíticas como tablas independientes. Así quedan disponibles para consultas, dashboards o ejercicios posteriores.

Con esto cerramos el flujo completo: origen externo, capa Bronze, refinamiento Silver y consumo analítico Gold.


In [20]:
gold_top_players.write.mode("overwrite").saveAsTable(
    "deepdivecatalog_ouro.ouro_top_players"
)

gold_value_by_country.write.mode("overwrite").saveAsTable(
    "deepdivecatalog_ouro.ouro_value_by_country"
)

gold_high_potential.write.mode("overwrite").saveAsTable(
    "deepdivecatalog_ouro.ouro_high_potential"
)

gold_efficiency.write.mode("overwrite").saveAsTable(
    "deepdivecatalog_ouro.ouro_efficiency"
)